# Análisis de Trayectorias Vehiculares mediante Aprendizaje No Supervisado
**Contexto:** Una empresa de logística cuenta con miles de registros de telemetría y trayectorias GPS de sus camiones (latitud, longitud, velocidad, aceleración, giros bruscos, hora del día), sin etiquetas previas sobre si un viaje fue "normal" o "peligroso". El objetivo de este cuaderno es aplicar técnicas de aprendizaje no supervisado para descubrir patrones ocultos, agrupar estilos de conducción y detectar anomalías críticas (por ejemplo, desvíos de ruta por robo o conducción temeraria).

**Nota sobre los datos:** dado que no se proporcionó un dataset específico junto con la rúbrica, se generó un conjunto de datos sintético (`telemetria_flota.csv`) que simula 40 camiones con 60 registros de telemetría cada uno (2400 registros en total), replicando tres estilos de conducción plausibles (eficiente, urbano, carretera) y un ~4% de anomalías críticas inyectadas deliberadamente (desvíos de ruta fuera de horario y conducción temeraria a alta velocidad).

**Algoritmo de clustering seleccionado:** conforme a lo solicitado, se desarrolla el punto 3 de la rúbrica utilizando **únicamente K-Means** (no se implementa DBSCAN en este cuaderno).

In [1]:
#Librerías necesarias para esta actividad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.ensemble import IsolationForest

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8,5)
np.random.seed(42)
pd.set_option("display.max_columns", None)
print("Librerías instaladas...")

Librerías instaladas...


In [ ]:
#Generación del Dataset
n_trucks = 40
records_per_truck = 60
rows = []

profiles = {
    "eficiente": dict(vel=(45,10), acc=(0.3,0.15), giros=(1,1), hora=(9,15)),
    "urbano":    dict(vel=(25,8),  acc=(0.6,0.2),  giros=(4,2), hora=(8,16)),
    "carretera": dict(vel=(80,12), acc=(0.4,0.15), giros=(1,1), hora=(6,20)),
}
profile_names = list(profiles.keys())
base_lat, base_lon = -0.9, -80.7

truck_id = 0
for t in range(n_trucks):
    truck_id += 1
    profile = np.random.choice(profile_names, p=[0.35,0.35,0.30])
    p = profiles[profile]
    route_lat = base_lat + np.random.uniform(-1.5,1.5)
    route_lon = base_lon + np.random.uniform(-1.5,1.5)
    for r in range(records_per_truck):
        vel = max(0, np.random.normal(p["vel"][0], p["vel"][1]))
        acc = max(0, np.random.normal(p["acc"][0], p["acc"][1]))
        giros = max(0, np.random.poisson(p["giros"][0]))
        hora = np.clip(np.random.normal(p["hora"][0], p["hora"][1]), 0, 23)
        lat = route_lat + np.random.normal(0, 0.05)
        lon = route_lon + np.random.normal(0, 0.05)
        rows.append([f"CAM-{truck_id:03d}", profile, lat, lon, vel, acc, giros, hora])

df = pd.DataFrame(rows, columns=["vehiculo_id","perfil_real","latitud","longitud",
                                  "velocidad","aceleracion","giros_bruscos","hora_dia"])

# Inyección de anomalías reales (4%): desvío de ruta / robo y conducción temeraria
n_anom = int(len(df)*0.04)
anom_idx = np.random.choice(df.index, n_anom, replace=False)
for i in anom_idx:
    kind = np.random.choice(["desvio","temeraria"])
    if kind == "desvio":
        df.loc[i, "latitud"] += np.random.uniform(0.8,1.5)*np.random.choice([-1,1])
        df.loc[i, "longitud"] += np.random.uniform(0.8,1.5)*np.random.choice([-1,1])
        df.loc[i, "hora_dia"] = np.random.uniform(0,4)
        df.loc[i, "velocidad"] = np.random.uniform(0,10)
    else:
        df.loc[i, "velocidad"] = np.random.uniform(120,160)
        df.loc[i, "aceleracion"] = np.random.uniform(2.5,4)
        df.loc[i, "giros_bruscos"] = np.random.randint(10,20)
df.loc[anom_idx, "es_anomalia_real"] = 1
df["es_anomalia_real"] = df["es_anomalia_real"].fillna(0).astype(int)

# Registros de ruido de sensores: nulos
n_null = 30
null_idx = np.random.choice(df.index, n_null, replace=False)
for i in null_idx:
    col = np.random.choice(["velocidad","aceleracion","giros_bruscos"])
    df.loc[i, col] = np.nan

df = df.sample(frac=1, random_state=1).reset_index(drop=True)
df.to_csv("telemetria_flota.csv", index=False)
print("Dimensiones del dataset:", df.shape)
df.head()